## EDA

In [0]:
from pyspark.sql import functions as F

In [0]:
df = spark.sql("""
SELECT *
FROM resume_project.gold_layer.gold_master
""")
display(df)

sales_id,quantity,discount,total_amount,date,customer_id,name,email,email_domain,location,signup_date,product_id,product_name,category,price,store_id,store_name,region
68,1,19.53,83.56,2025-06-12,81,stephanie foster,hcantu@parker.com,parker,port lisaville,2024-10-11,27,Also Term,Clothing,103.84,5,Store_5,EAST
73,2,21.77,83.47,2025-03-12,54,christopher williams,ejones@owens.info,owens,east rebecca,2023-08-18,38,Official Reality,Clothing,53.35,14,Store_14,NORTH
84,1,24.3,237.34,2025-04-06,68,david kennedy,woodsjamie@yahoo.com,yahoo,gloverstad,2025-02-28,82,Significant Them,Toys,313.53,18,Store_18,WEST
110,4,14.44,1386.62,2025-01-06,148,jeffrey henson,alexanderwagner@yahoo.com,yahoo,lake tammy,2025-02-28,36,Republican Will,Electronics,405.16,8,Store_8,WEST
120,3,14.95,785.53,2025-02-10,129,bruce newman,diazstephanie@hotmail.com,hotmail,north stacy,2023-10-17,21,Want Value,Furniture,307.87,14,Store_14,NORTH
130,1,0.43,480.91,2024-08-08,90,jeffrey fisher,eddie87@miller.biz,miller,lewistown,2023-12-16,35,School Support,Toys,482.99,16,Store_16,NORTH
137,3,29.01,71.88,2025-05-01,140,catherine lopez,samuel81@herrera.org,herrera,south eugeneview,2025-05-06,7,Today Want,Clothing,33.75,2,Store_2,WEST
155,4,24.24,903.82,2025-06-06,200,danielle parker,natalie37@burns.org,burns,duncanfurt,2023-10-02,29,Player Type,Electronics,298.25,12,Store_12,EAST
157,1,15.44,62.62,2024-09-12,166,pamela nguyen,darrenjacobson@yahoo.com,yahoo,millerstad,2022-11-13,22,Any Very,Books,74.05,15,Store_15,WEST
163,1,0.01,432.2,2024-12-02,93,jeffrey schultz,sandracarpenter@neal.com,neal,east ericview,2025-03-16,81,Summer Peace,Furniture,432.24,12,Store_12,EAST


In [0]:
df = df.withColumnRenamed("date", "order_date")
display(df)

sales_id,quantity,discount,total_amount,order_date,customer_id,name,email,email_domain,location,signup_date,product_id,product_name,category,price,store_id,store_name,region
68,1,19.53,83.56,2025-06-12,81,stephanie foster,hcantu@parker.com,parker,port lisaville,2024-10-11,27,Also Term,Clothing,103.84,5,Store_5,EAST
73,2,21.77,83.47,2025-03-12,54,christopher williams,ejones@owens.info,owens,east rebecca,2023-08-18,38,Official Reality,Clothing,53.35,14,Store_14,NORTH
84,1,24.3,237.34,2025-04-06,68,david kennedy,woodsjamie@yahoo.com,yahoo,gloverstad,2025-02-28,82,Significant Them,Toys,313.53,18,Store_18,WEST
110,4,14.44,1386.62,2025-01-06,148,jeffrey henson,alexanderwagner@yahoo.com,yahoo,lake tammy,2025-02-28,36,Republican Will,Electronics,405.16,8,Store_8,WEST
120,3,14.95,785.53,2025-02-10,129,bruce newman,diazstephanie@hotmail.com,hotmail,north stacy,2023-10-17,21,Want Value,Furniture,307.87,14,Store_14,NORTH
130,1,0.43,480.91,2024-08-08,90,jeffrey fisher,eddie87@miller.biz,miller,lewistown,2023-12-16,35,School Support,Toys,482.99,16,Store_16,NORTH
137,3,29.01,71.88,2025-05-01,140,catherine lopez,samuel81@herrera.org,herrera,south eugeneview,2025-05-06,7,Today Want,Clothing,33.75,2,Store_2,WEST
155,4,24.24,903.82,2025-06-06,200,danielle parker,natalie37@burns.org,burns,duncanfurt,2023-10-02,29,Player Type,Electronics,298.25,12,Store_12,EAST
157,1,15.44,62.62,2024-09-12,166,pamela nguyen,darrenjacobson@yahoo.com,yahoo,millerstad,2022-11-13,22,Any Very,Books,74.05,15,Store_15,WEST
163,1,0.01,432.2,2024-12-02,93,jeffrey schultz,sandracarpenter@neal.com,neal,east ericview,2025-03-16,81,Summer Peace,Furniture,432.24,12,Store_12,EAST


In [0]:
df.createOrReplaceTempView("gold_sales")

### A. Sales Overview

In [0]:
##total revenue
t_rev = df.select(F.round(F.sum("total_amount"), 2)).collect()[0][0]

print(f"Total Revenue: ${t_rev}")

Total Revenue: $1010895.57


In [0]:
##Total transactions
total_txns = df.count()
print(f"Total Transactions: {total_txns}")

Total Transactions: 2000


In [0]:
##total quantity
t_quant= df.select(F.sum("quantity")).collect()[0][0]

print(f"Total Quantity Sold: {t_quant}")

Total Quantity Sold: 4960


In [0]:
##number of customers
custs = df.select(F.countDistinct("customer_id")).collect()[0][0]
print(f"Number of Unique Customers: {custs}")

Number of Unique Customers: 200


In [0]:
##number of products
prods = df.select(F.countDistinct("product_id")).collect()[0][0]
print(f"Number of Unique Products: {prods}")

Number of Unique Products: 100


In [0]:
##numer of stores
stores = df.select(F.countDistinct("store_id")).collect()[0][0]
print(f"Number of Unique Stores: {stores}")

Number of Unique Stores: 20


In [0]:
%sql
--time range of sales
SELECT
    MIN(order_date) AS first_sale_date,
    MAX(order_date) AS last_sale_date
FROM gold_sales

first_sale_date,last_sale_date
2024-06-28,2025-06-28


### B. Time Analysis

In [0]:
##daily sales
daily_sales = df.groupBy("order_date").agg(F.sum("total_amount").alias("daily_total_sales"))
display(daily_sales)

order_date,daily_total_sales
2025-06-12,83.56
2025-03-12,2316.6
2025-04-06,806.37
2025-01-06,3012.0699999999997
2025-02-10,1571.95
2024-08-08,2222.09
2025-05-01,1431.3
2025-06-06,2844.4700000000003
2024-09-12,2027.82
2024-12-02,1082.3400000000001


Databricks visualization. Run in Databricks to view.

In [0]:
##weekly sales
weekly_sales = df.groupBy(F.date_trunc("week", "order_date").alias("week")).agg(F.sum("total_amount").alias("weekly_total_sales"))
display(weekly_sales)


week,weekly_total_sales
2025-06-09T00:00:00.000Z,9297.93
2025-03-10T00:00:00.000Z,6580.86
2025-03-31T00:00:00.000Z,7981.51
2025-01-06T00:00:00.000Z,8521.62
2025-02-10T00:00:00.000Z,12905.86
2024-08-05T00:00:00.000Z,7962.33
2025-04-28T00:00:00.000Z,8635.99
2025-06-02T00:00:00.000Z,9506.940000000002
2024-09-09T00:00:00.000Z,9459.2
2024-12-02T00:00:00.000Z,7838.29


Databricks visualization. Run in Databricks to view.

In [0]:
##monthly sales
monthly_sales = df.groupBy(F.date_trunc("month", "order_date").alias("month")).agg(F.sum("total_amount").alias("monthly_total_sales"))
display(monthly_sales)

month,monthly_total_sales
2025-06-01T00:00:00.000Z,549752.3300000003
2025-03-01T00:00:00.000Z,35998.49000000001
2025-04-01T00:00:00.000Z,46703.78000000002
2025-01-01T00:00:00.000Z,42049.530000000006
2025-02-01T00:00:00.000Z,38061.520000000004
2024-08-01T00:00:00.000Z,45259.43000000001
2025-05-01T00:00:00.000Z,45636.16
2024-09-01T00:00:00.000Z,36343.61000000001
2024-12-01T00:00:00.000Z,41588.659999999996
2024-10-01T00:00:00.000Z,35575.01999999998


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
--seasonality trends
--monthly (years combined)
SELECT
    MONTH(order_date) AS month,
    ROUND(SUM(total_amount), 2) AS revenue,
    COUNT(sales_id) AS transactions
FROM gold_sales
GROUP BY MONTH(order_date)
ORDER BY month

month,revenue,transactions
1,42049.53,77
2,38061.52,87
3,35998.49,69
4,46703.78,92
5,45636.16,85
6,551838.57,1077
7,46693.86,92
8,45259.43,104
9,36343.61,82
10,35575.02,71


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
--seaonality by category
SELECT
    category,
    MONTH(order_date) AS month,
    ROUND(SUM(total_amount), 2) AS revenue
FROM gold_sales
GROUP BY category, MONTH(order_date)
ORDER BY category, month

category,month,revenue
Books,1,6396.3
Books,2,3346.62
Books,3,5440.55
Books,4,6824.65
Books,5,4196.06
Books,6,60247.92
Books,7,6781.59
Books,8,2312.1
Books,9,3217.0
Books,10,4452.56


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
--seasonality by region
SELECT
    region,
    MONTH(order_date) AS month,
    ROUND(SUM(total_amount), 2) AS revenue
FROM gold_sales
GROUP BY region, MONTH(order_date)
ORDER BY region, month

region,month,revenue
EAST,1,6738.17
EAST,2,12124.53
EAST,3,10489.15
EAST,4,11220.12
EAST,5,7984.18
EAST,6,145024.96
EAST,7,8330.96
EAST,8,8867.08
EAST,9,11558.25
EAST,10,9913.15


Databricks visualization. Run in Databricks to view.

### C. Product Analysis

In [0]:
##top products by revenue
top_prod_rev = df.groupBy("product_name").agg(F.sum("total_amount").alias("total_revenue")).orderBy(F.desc("total_revenue")).limit(10)
display(top_prod_rev)

product_name,total_revenue
Five Thing,29513.799999999992
Table Research,27723.340000000004
Friend Course,26838.389999999992
Operation Store,24700.949999999997
Summer Peace,21669.11
Technology Suffer,20986.960000000003
Necessary Some,20637.219999999998
Significant Them,20186.000000000004
Wish Actually,19325.639999999996
Contain Care,19314.62


In [0]:
##top products by quantity
top_prod_qut = df.groupBy("product_name").agg(F.sum("quantity").alias("total_quantity")).orderBy(F.desc("total_quantity")).limit(10)
display(top_prod_qut)

product_name,total_quantity
Table Research,85
Radio Hospital,82
Friend Course,80
Significant Them,77
Five Thing,76
Player Type,76
Game Race,74
Star Power,72
Operation Store,70
During Her,69


### D. Store Analysis

In [0]:
##top stores by revenue
top_st_rev = df.groupBy("store_name").agg(F.sum("total_amount").alias("total_revenue")).orderBy(F.desc("total_revenue")).limit(10)
display(top_st_rev)

store_name,total_revenue
Store_2,59304.63
Store_13,58421.329999999994
Store_20,58076.83999999999
Store_18,56539.76999999998
Store_4,56376.69000000001
Store_12,53873.169999999984
Store_10,53761.25000000001
Store_6,53556.39
Store_16,53470.40999999999
Store_3,51089.070000000014


In [0]:
##average basket size by store
avg_st_basket = df.groupBy("store_name").agg(F.round(F.avg("quantity"), 2).alias("avg_basket_size")).orderBy(F.desc("avg_basket_size"))
display(avg_st_basket)

store_name,avg_basket_size
Store_13,2.69
Store_14,2.67
Store_5,2.63
Store_20,2.59
Store_3,2.56
Store_15,2.55
Store_6,2.55
Store_7,2.53
Store_2,2.49
Store_16,2.49


In [0]:
##regional performance metrics
regional_perf = (
    df.groupBy("region")
      .agg(
          F.round(F.sum("total_amount"), 2).alias("total_revenue"),
          F.count("*").alias("transactions"),
          F.round(F.avg("total_amount"), 2).alias("avg_order_value"),
          F.round(F.avg("quantity"), 2).alias("avg_items_per_order")
      )
      .orderBy(F.desc("total_revenue"))
)

display(regional_perf)

region,total_revenue,transactions,avg_order_value,avg_items_per_order
WEST,400222.67,787,508.54,2.46
EAST,254210.63,520,488.87,2.52
SOUTH,204483.21,408,501.18,2.44
NORTH,151979.06,285,533.26,2.52


In [0]:
##regional market share
total_revenue = df.agg(F.sum("total_amount")).collect()[0][0]

regional_market_share = (
    df.groupBy("region")
      .agg(F.sum("total_amount").alias("total_revenue"))
      .withColumn(
          "market_share",
          F.round(F.col("total_revenue") / total_revenue, 2)
      )
      .orderBy(F.desc("market_share"))
)

display(regional_market_share)


region,total_revenue,market_share
WEST,400222.67000000016,0.4
EAST,254210.62999999995,0.25
SOUTH,204483.21000000002,0.2
NORTH,151979.06000000003,0.15


### E. Customer Analysis

In [0]:
##top customers by revenue
top_cust_rev = df.groupBy("customer_id").agg(F.sum("total_amount").alias("total_revenue")).orderBy(F.desc("total_revenue")).limit(10)
display(top_cust_rev)

customer_id,total_revenue
151,11934.03
4,10360.26
118,10181.51
22,9979.539999999999
113,9735.54
99,9512.919999999998
144,9287.2
32,9201.289999999999
65,9153.65
47,8702.509999999998


In [0]:
##repeat vs one-time buyers
cust_orders = df.groupBy("customer_id").agg(F.countDistinct("sales_id").alias("num_orders"))

repeat_cust = cust_orders.filter(F.col("num_orders") > 1)
one_time_cust = cust_orders.filter(F.col("num_orders") == 1)


print(f"Repeat customers: {repeat_cust.count()}")
print(f"One-time customers: {one_time_cust.count()}")

Repeat customers: 200
One-time customers: 0
